# Analyse de sentiment Hassaniya / Pulaar — **deux modèles spécialisés**

**Projet de Master — NLP pour langues peu dotées**

Stratégie : un modèle dédié par langue, car la tâche est *bi-script*.
- **Hassaniya** (écriture arabe) → `UBC-NLP/MARBERT` (arabe dialectal)
- **Pulaar** (écriture latine) → `Davlan/afro-xlmr-base` (langues africaines)

On découpe **au niveau de la phrase** (mêmes phrases de test pour les deux modèles, pas de fuite), on entraîne chaque modèle sur sa langue, puis on évalue : par langue, et en **système combiné** (chaque texte est routé vers le bon modèle).

Labels : `negatif`→0, `neutre`→1, `positif`→2.


In [ ]:
!pip install -q "transformers[torch]" datasets evaluate accelerate scikit-learn matplotlib seaborn gradio sentencepiece -U

## 1. Chargement et préparation

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

CSV_PATH = "/content/dataset_master (2).csv"   # adapte si besoin
df = pd.read_csv(CSV_PATH)

labeled = df[df["sentiment"].notna()].copy()
labeled["sentiment"] = labeled["sentiment"].astype(str).str.strip().str.lower()
label_map = {"negatif":0, "neutre":1, "positif":2}
labeled["label"] = labeled["sentiment"].map(label_map)
labeled = labeled.dropna(subset=["label"]).copy()
labeled["label"] = labeled["label"].astype(int)

id2label = {0:"Négatif", 1:"Neutre", 2:"Positif"}
label2id = {v:k for k,v in id2label.items()}

has_pulaar = labeled["pulaar"].notna() & (labeled["pulaar"].astype(str).str.strip()!="")
print("Phrases annotees :", len(labeled))
print("  - Hassaniya :", labeled["hassaniya_norm"].notna().sum())
print("  - Pulaar    :", has_pulaar.sum())
print(labeled["label"].map(id2label).value_counts())

## 2. Découpage au niveau de la phrase (80/10/10), puis extraction par langue

Le même découpage de phrases sert aux deux modèles : les phrases de test sont identiques, ce qui rend le score système combiné cohérent.

In [ ]:
from sklearn.model_selection import train_test_split

train_rows, temp_rows = train_test_split(labeled, test_size=0.2, random_state=42, stratify=labeled["label"])
val_rows, test_rows   = train_test_split(temp_rows, test_size=0.5, random_state=42, stratify=temp_rows["label"])
print(f"Phrases -> Train {len(train_rows)} | Val {len(val_rows)} | Test {len(test_rows)}")

def lang_frame(rows, col):
    sub = rows[rows[col].notna() & (rows[col].astype(str).str.strip()!="")].copy()
    sub["texte"] = sub[col].astype(str).str.strip()
    return sub[["texte","label"]].reset_index(drop=True)

# Hassaniya (toujours present)
hass = {"train": lang_frame(train_rows,"hassaniya_norm"),
        "val":   lang_frame(val_rows,"hassaniya_norm"),
        "test":  lang_frame(test_rows,"hassaniya_norm")}
# Pulaar (sous-ensemble qui possede une traduction)
pul  = {"train": lang_frame(train_rows,"pulaar"),
        "val":   lang_frame(val_rows,"pulaar"),
        "test":  lang_frame(test_rows,"pulaar")}

for name, d in [("HASSANIYA", hass), ("PULAAR", pul)]:
    print(f"{name:10s} -> train {len(d['train'])} | val {len(d['val'])} | test {len(d['test'])}")

## 3. Baseline par langue (TF-IDF + Régression logistique)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

def baseline(d, nom):
    vec = TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=20000)
    Xtr = vec.fit_transform(d["train"]["texte"]); Xte = vec.transform(d["test"]["texte"])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, d["train"]["label"])
    pred = clf.predict(Xte)
    f1 = f1_score(d["test"]["label"], pred, average="macro", zero_division=0)
    print(f"=== BASELINE {nom} === F1-macro = {f1:.4f}")
    return f1

baseline_hass = baseline(hass, "HASSANIYA")
baseline_pul  = baseline(pul,  "PULAAR")

## 4. Fonction d'entraînement générique (un modèle par langue)

Avec poids de classe, F1-macro comme critère, et arrêt anticipé. Le même code sert pour MARBERT et AfroXLMR (le tokeniseur s'adapte automatiquement à chaque modèle).

In [ ]:
import torch
from torch import nn
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.metrics import accuracy_score, precision_score, recall_score

MAX_LEN = 64

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
            "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
            "recall_macro": recall_score(labels, preds, average="macro", zero_division=0)}

def class_weights_from(y):
    from sklearn.utils.class_weight import compute_class_weight
    present = np.unique(y)
    w = compute_class_weight("balanced", classes=present, y=y)
    wmap = {int(c): float(wi) for c, wi in zip(present, w)}
    return torch.tensor([wmap.get(i, 1.0) for i in range(3)], dtype=torch.float)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw); self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs); logits = out.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = nn.CrossEntropyLoss(weight=w)(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, out) if return_outputs else loss

def train_lang(model_name, d, tag, batch=16, epochs=15):
    tok = AutoTokenizer.from_pretrained(model_name)
    def prep(ex): return tok(ex["texte"], padding="max_length", truncation=True, max_length=MAX_LEN)
    dd = DatasetDict({k: Dataset.from_pandas(d[k][["texte","label"]], preserve_index=False)
                      for k in ["train","val","test"]})
    dd = dd.map(prep, batched=True, remove_columns=[c for c in dd["train"].column_names if c!="label"])
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3, id2label=id2label, label2id=label2id)
    args = TrainingArguments(
        output_dir=f"./res_{tag}", eval_strategy="epoch", save_strategy="epoch",
        num_train_epochs=epochs, per_device_train_batch_size=batch, per_device_eval_batch_size=batch,
        learning_rate=2e-5, weight_decay=0.01, load_best_model_at_end=True,
        metric_for_best_model="f1_macro", greater_is_better=True, logging_steps=20, report_to="none")
    trainer = WeightedTrainer(model=model, args=args, train_dataset=dd["train"],
        eval_dataset=dd["val"], compute_metrics=compute_metrics,
        class_weights=class_weights_from(d["train"]["label"].values),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
    print(f"\n>>> Entrainement {tag} ({model_name})")
    trainer.train()
    return tok, trainer, dd

## 5. Modèle 1 — Hassaniya avec MARBERT

In [ ]:
MODEL_HASS = "UBC-NLP/MARBERT"
tok_hass, trainer_hass, dd_hass = train_lang(MODEL_HASS, hass, "hass")

## 6. Modèle 2 — Pulaar avec AfroXLMR

AfroXLMR est plus volumineux : si la mémoire GPU sature, repasse `batch=8`.

In [ ]:
MODEL_PUL = "Davlan/afro-xlmr-base"
tok_pul, trainer_pul, dd_pul = train_lang(MODEL_PUL, pul, "pul", batch=16)

## 7. Évaluation par langue

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
target_names = ["Négatif","Neutre","Positif"]

def evaluer(trainer, dd, d, nom, baseline_f1):
    pred = trainer.predict(dd["test"])
    yp = np.argmax(pred.predictions, axis=-1); yt = pred.label_ids
    print(f"=== {nom} (TEST) ===")
    print(classification_report(yt, yp, target_names=target_names, zero_division=0))
    f1 = f1_score(yt, yp, average="macro", zero_division=0)
    print(f"F1-macro {nom} : {f1:.4f}  (baseline {baseline_f1:.4f}, gain {f1-baseline_f1:+.4f})\n")
    return yt, yp, f1

yt_h, yp_h, f1_h = evaluer(trainer_hass, dd_hass, hass, "HASSANIYA (MARBERT)", baseline_hass)
yt_p, yp_p, f1_p = evaluer(trainer_pul,  dd_pul,  pul,  "PULAAR (AfroXLMR)",  baseline_pul)

In [ ]:
# Matrices de confusion cote a cote
fig, axes = plt.subplots(1, 2, figsize=(11,5))
for ax, (yt, yp, titre) in zip(axes, [(yt_h,yp_h,"Hassaniya - MARBERT"), (yt_p,yp_p,"Pulaar - AfroXLMR")]):
    ConfusionMatrixDisplay(confusion_matrix(yt, yp), display_labels=target_names).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(titre)
plt.tight_layout(); plt.show()

## 8. Score SYSTÈME combiné

On agrège les prédictions des deux modèles : chaque texte de test est traité par le modèle de sa langue. Ce F1-macro global est le chiffre comparable au notebook mono-modèle (mBERT).

In [ ]:
y_true_all = np.concatenate([yt_h, yt_p])
y_pred_all = np.concatenate([yp_h, yp_p])
print("=== SYSTEME COMBINE (Hassaniya + Pulaar) ===")
print(classification_report(y_true_all, y_pred_all, target_names=target_names, zero_division=0))
f1_systeme = f1_score(y_true_all, y_pred_all, average="macro", zero_division=0)
print(f"F1-macro SYSTEME = {f1_systeme:.4f}")
print(f"(Rappel: mBERT mono-modele faisait ~0.40 sur le meme test)")

# Tableau recapitulatif
recap = pd.DataFrame({
    "Composant": ["Hassaniya (MARBERT)", "Pulaar (AfroXLMR)", "Systeme combine"],
    "F1-macro":  [round(f1_h,4), round(f1_p,4), round(f1_systeme,4)],
    "Baseline":  [round(baseline_hass,4), round(baseline_pul,4), "-"],
})
print("\n"); print(recap.to_string(index=False))

## 9. Comparaison visuelle

In [ ]:
labels = ["Hassaniya", "Pulaar", "Systeme"]
modele = [f1_h, f1_p, f1_systeme]
base   = [baseline_hass, baseline_pul, np.nan]
x = np.arange(len(labels)); w = 0.35
plt.figure(figsize=(7,4))
plt.bar(x-w/2, modele, w, label="Modeles specialises")
plt.bar(x+w/2, base, w, label="Baseline TF-IDF")
plt.xticks(x, labels); plt.ylabel("F1-macro"); plt.ylim(0,1)
plt.title("Modeles specialises vs baseline"); plt.legend()
for i,v in enumerate(modele): plt.text(i-w/2, v+0.02, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout(); plt.show()

## 10. Analyse d'erreurs (par langue)

In [ ]:
def erreurs(d, yt, yp, nom):
    e = pd.DataFrame({"texte": d["test"]["texte"].values, "vrai":[id2label[i] for i in yt],
                      "predit":[id2label[i] for i in yp]})
    e = e[e["vrai"]!=e["predit"]]
    print(f"--- {nom} : {len(e)} erreurs / {len(yt)} ---")
    return e

pd.set_option("display.max_colwidth", 90)
err_h = erreurs(hass, yt_h, yp_h, "HASSANIYA"); display(err_h.head(10))
err_p = erreurs(pul,  yt_p, yp_p, "PULAAR");    display(err_p.head(10))

## 11. Sauvegarde des deux modèles

In [ ]:
trainer_hass.save_model("./modele_hassaniya"); tok_hass.save_pretrained("./modele_hassaniya")
trainer_pul.save_model("./modele_pulaar");        tok_pul.save_pretrained("./modele_pulaar")
print("Modeles sauvegardes : ./modele_hassaniya et ./modele_pulaar")

## 12. Démo Gradio — routage automatique selon l'écriture

Si le texte contient des caractères arabes → modèle Hassaniya (MARBERT). Sinon → modèle Pulaar (AfroXLMR). Un menu permet aussi de forcer la langue.

In [ ]:
import gradio as gr, re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m_hass = trainer_hass.model.to(device).eval()
m_pul  = trainer_pul.model.to(device).eval()

def detecter_langue(texte):
    return "Hassaniya" if re.search(r"[\u0600-\u06FF]", texte) else "Pulaar"

def analyser(texte, choix_langue):
    try:
        langue = detecter_langue(texte) if choix_langue=="Automatique" else choix_langue
        tok, model = (tok_hass, m_hass) if langue=="Hassaniya" else (tok_pul, m_pul)
        inputs = tok(texte, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(device)
        with torch.no_grad(): probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
        noms = ["Négatif 😠","Neutre 😐","Positif 😊"]
        res = {noms[i]: float(probs[i]) for i in range(3)}
        return f"Langue : {langue}", res
    except Exception as e:
        return "Erreur", {"Erreur": str(e)}

demo = gr.Interface(
    fn=analyser,
    inputs=[gr.Textbox(lines=2, label="Avis citoyen", placeholder="Tapez en Hassaniya ou Pulaar..."),
            gr.Radio(["Automatique","Hassaniya","Pulaar"], value="Automatique", label="Langue")],
    outputs=[gr.Textbox(label="Modèle utilisé"), gr.Label(num_top_classes=3, label="Sentiment")],
    title="Analyse des Avis Citoyens — Système bilingue (2 modèles spécialisés)",
    description="MARBERT pour le Hassaniya, AfroXLMR pour le Pulaar.",
    examples=[
        ["هي عدلت مارو للعشا.", "Automatique"],
        ["o defi ko maro hirandé", "Automatique"],
        ["اينيتي وحل فلباسي", "Automatique"],
        ["Gaɗi ina nanngi e comci am", "Automatique"],
    ],
)
demo.launch(share=True, debug=True)